# Tuning - Random Forest Regressor para limiar de reposicao

Este notebook desenvolve o sucessor do artefato historico `03_tree_ensembles_random_forest_regressor_threshold_model.pkl`, mas sem salvar `.pkl` localmente. O modelo, metricas, predicoes, parametros e graficos sao registrados diretamente no MLflow.


## Estrategia

Usamos `TimeSeriesSplit`, uma validacao cruzada temporal. Ela preserva a ordem passado -> futuro e evita vazamento temporal, que ocorreria com K-Fold aleatorio. O conjunto `test` fica isolado para avaliacao final do campeao.

O tuning usa `RandomizedSearchCV` porque o grid completo seria caro para florestas com varias combinacoes de profundidade, folhas e numero de arvores. Alem dos hiperparametros, cada busca e repetida com perfis de peso diferentes para testar custos de erro diferentes.


In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR is None:
    raise RuntimeError("Nao encontrei ml/notebooks/utils. Execute o notebook dentro do repositorio Saltim.")
sys.path.insert(0, str(UTILS_DIR))

import pandas as pd

from regressor_tuning_common import (
    RegressorTuningConfig,
    WEIGHT_PROFILE_DESCRIPTIONS,
    run_regressor_tuning,
)

from sklearn.ensemble import RandomForestRegressor


In [2]:
RF_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 6, 10, 16, 24],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8, 1.0],
    "model__bootstrap": [True],
}

def rf_model_factory() -> RandomForestRegressor:
    return RandomForestRegressor(random_state=42, n_jobs=-1)

def rf_baseline_factory() -> RandomForestRegressor:
    return RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

config = RegressorTuningConfig(
    notebook_id="05_random_forest_regressor_threshold_tuning",
    family="Arvores",
    model_name="Random Forest Regressor",
    model_factory=rf_model_factory,
    baseline_factory=rf_baseline_factory,
    param_distributions=RF_PARAM_DISTRIBUTIONS,
    n_iter=16,
    cv_splits=4,
    random_state=42,
    use_full_dataset=True,
)

search_space = pd.DataFrame(
    [{"parametro": key, "valores": values} for key, values in RF_PARAM_DISTRIBUTIONS.items()]
)
weight_profiles = pd.DataFrame(
    [{"perfil": key, "descricao": value} for key, value in WEIGHT_PROFILE_DESCRIPTIONS.items()]
)

display(search_space)
display(weight_profiles)


,parametro,valores
0,model__n_estimators,"[100, 200, 300, 500]"
1,model__max_depth,"[None, 6, 10, 16, 24]"
2,model__min_samples_split,"[2, 5, 10, 20]"
3,model__min_samples_leaf,"[1, 2, 4, 8]"
4,model__max_features,"[sqrt, log2, 0.5, 0.8, 1.0]"
5,model__bootstrap,[True]


,perfil,descricao
0,uniform,Peso 1 para todas as amostras; usado como cont...
1,alert_focus,Aumenta peso de observacoes em Alerta de compr...
2,critical_gap_focus,Aumenta peso quando cobertura esta abaixo/prox...
3,threshold_extreme_focus,Aumenta peso de limiares de alerta mais distan...


## Execucao do tuning

A celula abaixo executa baseline, buscas randomicas por perfil de peso, registro dos candidatos no MLflow, treino final do campeao e diagnosticos de ajuste. Se o tempo estiver alto, reduza `n_iter` mantendo os mesmos perfis de peso.


In [3]:
results = run_regressor_tuning(config)

🏃 View run Random Forest Regressor baseline at: http://localhost:5000/#/experiments/17/runs/737b80de390647508a93090c3bf04ceb
🧪 View experiment at: http://localhost:5000/#/experiments/17
Fitting 4 folds for each of 16 candidates, totalling 64 fits


/home/sophia/www/cesar/6/projetos/Projetos-6-Saltim/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run Random Forest Regressor random search - uniform at: http://localhost:5000/#/experiments/17/runs/dee29b5d896f47b9bf50d21662c02723
🧪 View experiment at: http://localhost:5000/#/experiments/17
🏃 View run Random Forest Regressor tuning parent at: http://localhost:5000/#/experiments/17/runs/f917bdd5515a41259c7b27bd77677953
🧪 View experiment at: http://localhost:5000/#/experiments/17


KeyboardInterrupt: 

## Resultados quantitativos

As tabelas abaixo mostram baseline, melhores candidatos, variancia entre folds e metricas finais no teste.


In [ ]:
display(results["baseline_summary"])
display(results["candidate_results"].head(15))
display(results["best_fold_metrics"])
display(results["final_metrics"].T)
print("Melhor perfil de peso:", results["best_weight_profile"])
print("Melhores hiperparametros:", results["best_params"])
print("Diagnostico:", results["fit_diagnosis"])


## Visualizacoes de ajuste

A curva de aprendizado ajuda a identificar overfitting ou underfitting. A analise de residuos mostra vieses e dispersao dos erros no limiar previsto.


In [ ]:
results["figures"]["learning_curve"]


In [ ]:
results["figures"]["residual_analysis"]


## MLflow

No MLflow, procure pelos runs do experimento `notebooks/02_modelos_finais/05_random_forest_regressor_threshold_tuning/threshold_regression`. Os runs de candidatos guardam combinacoes de parametros e pesos; o run `champion` guarda o modelo registrado, predicoes, metricas finais e graficos.
